# 03 — Preprocessing Pipeline & Dataset Engineering

## Objective

This notebook upgrades the exploratory preprocessing pipeline developed in Notebook 02 into a reusable engineering workflow.

The goal is to build a preprocessing system that is:

- modular
- reusable
- training-ready
- robust against dataset inconsistencies

---

## What We Will Implement

This notebook introduces:

1. Clinical label engineering
2. Robust preprocessing utilities
3. Transform pipeline design
4. Production-style Dataset class
5. DataLoader engineering
6. Batch validation
7. GPU preprocessing validation

---

## Final Goal

Create a dataset pipeline that directly produces:

```python
{
    "video": tensor,
    "ef": float,
    "severity": int,
    "video_name": str,
    "split": str
}
```

This structure will later support:

- EfficientNet encoder
- multimodal fusion
- severity classification
- EF regression
- training pipeline

# 1. Imports & Environment Setup

## Concept

A preprocessing notebook typically centralizes:

- imports
- random seeds
- device configuration
- reproducibility settings

Keeping these definitions in one place improves:
- experiment consistency
- debugging
- reproducibility

In [ ]:
import cv2
import random
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from torch.utils.data import Dataset, DataLoader

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [ ]:

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# 2. Dataset Configuration

## Concept

Centralized path configuration improves:

- maintainability
- portability
- cleaner notebook structure

All dataset locations are defined once and reused throughout the notebook.

In [ ]:
from pathlib import Path
PROJECT_ROOT = Path(
    "/home/vasanth/projects/cardiac_vr_project"
)
DATA_DIR = PROJECT_ROOT / "data"
ECHONET_DIR = DATA_DIR / "EchoNet-Dynamic"
VIDEOS_DIR = ECHONET_DIR / "Videos"
FILELIST_PATH = ECHONET_DIR / "FileList.csv"
TRACINGS_PATH = ECHONET_DIR / "VolumeTracings.csv"
# CAMUS_DIR = DATA_DIR / "CAMUS" / "database_nifti"
MODEL_DIR = PROJECT_ROOT / "models"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

In [ ]:
print(VIDEOS_DIR.exists())
print(FILELIST_PATH.exists())
print(TRACINGS_PATH.exists())
# print(CAMUS_DIR.exists())

# 3. Metadata Loading & Validation

## Concept

Before preprocessing begins, dataset metadata must be validated.

This stage verifies:

- dataset accessibility
- metadata integrity
- annotation availability

Early validation prevents downstream debugging issues.

In [ ]:
filelist = pd.read_csv(FILELIST_PATH)
tracings = pd.read_csv(TRACINGS_PATH)

In [ ]:
filelist.head()

In [ ]:
tracings.head()

In [ ]:
print("FileList Shape :", filelist.shape)
print("Tracings Shape :", tracings.shape)

In [ ]:
filelist.describe()

In [ ]:
filelist.isnull().sum()

In [ ]:
import matplotlib.pyplot as plt
filelist.hist(figsize=(10,10))
plt.show()

In [ ]:
required_filelist_cols = {"FileName", "EF", "Split", "ESV", "EDV"}
required_tracing_cols  = {"FileName", "Frame", "X1", "Y1", "X2", "Y2"}

In [ ]:
missing_filelist = (required_filelist_cols -set(filelist.columns))
assert len(missing_filelist) == 0, f"Missing columns: {missing_filelist}"

In [ ]:
# Verify Tracing schema

missing_tracing = (required_tracing_cols - set(tracings.columns))
assert len(missing_tracing) == 0, f"Missing columns: {missing_tracing}"
print("Metadata validation successful.")

# 4. Clinical Label Engineering

## Concept

The project contains two predictive targets:

### EF Regression

Continuous target.

Example:

```text
EF = 57.0
```

---

### Severity Classification

Project-defined 3-class formulation:

```text
Normal
Mild
Severe
```

These labels will later support:

- classification training
- evaluation metrics
- multimodal fusion outputs

In [ ]:
def ef_to_severity(ef):
    if ef >= 50:
        return "Normal"
    elif ef >= 40:
        return "Mild"
    else:
        return "Severe"

In [ ]:

SEVERITY_MAP = {"Normal":0,"Mild":1,"Severe":2}

In [ ]:

filelist["Severity"] = (filelist["EF"].apply(ef_to_severity))
filelist["SeverityID"] = (filelist["Severity"].map(SEVERITY_MAP))

In [ ]:
filelist.head()

In [ ]:

filelist["Severity"].value_counts()

In [ ]:

plt.figure(figsize=(4,4))
filelist["Severity"].value_counts().plot(kind="pie")
plt.title("Severity Class Distribution")
plt.xlabel("Severity Class")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.show()

# 4.1 Clinical Metadata Validation

## Concept

In addition to EF and severity labels,
EchoNet provides cardiac volume measurements.

These measurements include:

- EDV (End Diastolic Volume)
- ESV (End Systolic Volume)

These variables may later support:
- multimodal analysis
- clinical interpretation
- auxiliary prediction tasks

In [ ]:

clinical_cols = ["EF","EDV","ESV","Severity","SeverityID"]
print(filelist[clinical_cols].head())

# 5. Video Validation Utilities

## Concept

Real-world medical datasets often contain:

- missing videos
- corrupted files
- invalid frame counts
- path inconsistencies

Before preprocessing begins, videos should be validated.

This stage improves:

- robustness
- debugging
- dataset reliability

In [ ]:

def validate_video(video_path):
    if not video_path.exists():
        return False, "File does not exist"
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return False, "OpenCV failed to open video"
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    if frame_count == 0:
        return False, "Zero frames detected"
    return True, "Video validated"

In [ ]:

def validate_row(sample_row):
    video_name = str(sample_row["FileName"]).strip()
    video_file = video_name + ".avi"
    video_path = VIDEOS_DIR / video_file
    status, message = validate_video(video_path)
    return video_name,video_path,status,message

# 6. Annotated Frame Discovery

## Concept

EchoNet provides frame-level tracing annotations.

These annotations help identify candidate cardiac phases.

This utility discovers frames containing contour information.

Later these frames will be used for:

- ED extraction
- ES extraction
- contour visualization

In [ ]:
def get_annotated_frames(tracings_df, video_name):
    """
    Returns frame indices annotated for a given video.
    EchoNet annotates exactly 2 frames per video:
      - ED frame (End Diastole — maximum volume)
      - ES frame (End Systole — minimum volume)

    Returns a Series of frame indices sorted by annotation frequency.
    """
    possible_names = [video_name, video_name + ".avi"]
    rows = tracings_df[tracings_df["FileName"].isin(possible_names)]

    if rows.empty:
        return pd.Series(dtype=int)

    frame_counts = rows["Frame"].value_counts().sort_values(ascending=False)
    return frame_counts

In [ ]:
video_name,video_path,status,message = validate_row(filelist.iloc[0])
print(video_name)
frame_counts = get_annotated_frames(tracings,video_name)
frame_counts.head()

# 7. Contour Construction Utilities

## Concept

EchoNet tracings contain two ventricular boundaries:

- X1,Y1
- X2,Y2

Both sides are required to reconstruct a complete contour.

Contour visualization is useful for:

- annotation validation
- ED/ES inspection
- clinical interpretability

In [ ]:
def build_contour(points_df):
    """
    Builds closed LV contour and extracts geometric features.
    Returns: contour array + feature dict
    """
    left_pts  = points_df[["X1", "Y1"]].values
    right_pts = points_df[["X2", "Y2"]].values[::-1]
    contour   = np.concatenate([left_pts, right_pts])

    x = contour[:, 0]
    y = contour[:, 1]

    # Area via Shoelace formula
    area = 0.5 * abs(
        np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1))
    )

    # Perimeter
    diffs     = np.diff(contour, axis=0, append=contour[:1])
    perimeter = np.sum(np.sqrt((diffs ** 2).sum(axis=1)))

    # Fit ellipse → major/minor axes
    contour_int = contour.astype(np.float32).reshape(-1, 1, 2)
    if len(contour_int) >= 5:
        ellipse    = cv2.fitEllipse(contour_int)
        major_axis = max(ellipse[1])
        minor_axis = min(ellipse[1])
    else:
        major_axis = np.sqrt(area)
        minor_axis = np.sqrt(area)

    # Compactness = 4π×area / perimeter²
    compactness  = (4 * np.pi * area) / (perimeter ** 2 + 1e-6)
    aspect_ratio = major_axis / (minor_axis + 1e-6)

    features = {
        "area":        float(area),
        "perimeter":   float(perimeter),
        "major_axis":  float(major_axis),
        "minor_axis":  float(minor_axis),
        "aspect_ratio": float(aspect_ratio),
        "compactness": float(compactness)
    }

    return contour, features

In [ ]:
def draw_contour(
    frame,
    contour_points,
    features=None,
    label=""
):
    """
    Draw LV contour and contour-derived features.

    Args:
        frame          : RGB image
        contour_points : (N,2) contour array
        features       : dict returned by build_contour()
        label          : ED / ES
    """

    frame_copy = frame.copy()

    contour_int = contour_points.astype(
        np.int32
    )

    # Draw contour
    cv2.polylines(
        frame_copy,
        [contour_int],
        isClosed=True,
        color=(0,255,0),
        thickness=2,
        lineType=cv2.LINE_AA
    )

    # Centroid from contour
    cx = int(contour_points[:,0].mean())
    cy = int(contour_points[:,1].mean())

    cv2.circle(frame_copy,(cx,cy),4,(255,0,0),-1)
    y = 20

    if label:
        cv2.putText(frame_copy,label,(10,y),cv2.FONT_HERSHEY_SIMPLEX,0.5,(255,255,0),1,cv2.LINE_AA)
        y += 20

    if features is not None:
        display_lines = [
            f"Area: {features['area']:.0f}",
            f"Major: {features['major_axis']:.1f}",
            f"Minor: {features['minor_axis']:.1f}",
            f"AR: {features['aspect_ratio']:.2f}",
            f"Compact: {features['compactness']:.2f}"
        ]

        for txt in display_lines:
            cv2.putText(frame_copy,txt,(10,y),cv2.FONT_HERSHEY_SIMPLEX,0.45,(255,255,0),1,cv2.LINE_AA)
            y += 18

    return frame_copy

In [ ]:
# Select one video
video_name = filelist.iloc[0]["FileName"]

# Get annotated frames
frame_counts = get_annotated_frames(
    tracings,
    video_name
)

print(frame_counts)

# Pick first annotated frame
frame_idx = frame_counts.index[0]

# Get tracing points for that frame
points_df = tracings[
    (tracings["FileName"].isin(
        [video_name, video_name + ".avi"]
    ))
    &
    (tracings["Frame"] == frame_idx)
]

# Build contour + features
contour, features = build_contour(
    points_df
)

print(features)

# Load frame from video
video_path = VIDEOS_DIR / f"{video_name}.avi"

cap = cv2.VideoCapture(str(video_path))

cap.set(
    cv2.CAP_PROP_POS_FRAMES,
    frame_idx
)

ret, frame = cap.read()

cap.release()

frame = cv2.cvtColor(
    frame,
    cv2.COLOR_BGR2RGB
)

# Draw contour + features
vis_frame = draw_contour(
    frame,
    contour
)

# Display
plt.figure(figsize=(4,4))

plt.imshow(vis_frame)

plt.axis("off")

plt.show()

# 8. Frame Preprocessing Pipeline

## Concept

Deep learning models require consistent tensor inputs.

This preprocessing function standardizes:

- resizing
- normalization
- tensor conversion
- channel ordering

Target tensor format:

(C, H, W)

In [ ]:
def preprocess_frame(frame, target_size=(224,224)):

    frame = cv2.resize(frame,target_size)
    frame = frame.astype(np.float32) / 255.0
    frame = torch.tensor(frame,dtype=torch.float32)
    frame = frame.permute(2,0,1)
    mean = torch.tensor([0.485,0.456,0.406]).view(3,1,1)
    std = torch.tensor([0.229,0.224,0.225]).view(3,1,1)
    frame = (frame - mean) / std
    return frame

# 9. Transform System

## Concept

Training and validation pipelines often use different preprocessing rules.

Examples:

Training:
- augmentation
- randomization

Validation:
- deterministic transforms

Separating transforms improves:

- reproducibility
- experimentation
- pipeline modularity

In [ ]:
def train_transform(frame):
    frame = preprocess_frame(frame)
    return frame

In [ ]:

def val_transform(frame):
    # Validation preprocessing remains deterministic
    frame = preprocess_frame(frame)
    return frame

# 10. Transform Validation

## Concept

Before integrating transforms into the dataset pipeline,
individual preprocessing stages should be validated independently.

In [ ]:
# reading video
cap = cv2.VideoCapture(str(video_path))
ret, frame = cap.read()
cap.release()


frame_rgb = cv2.cvtColor(frame,cv2.COLOR_BGR2RGB)

In [ ]:

sample_tensor = train_transform(frame_rgb)
print("Tensor Shape :",sample_tensor.shape)
print("Tensor Type  :",sample_tensor.dtype)

# 11. Production Dataset Class

## Concept

The Dataset class acts as the central preprocessing engine.

Its responsibilities include:

- metadata retrieval
- video loading
- ED/ES extraction
- preprocessing transforms
- label preparation

Unlike Notebook 02, this version returns both:

- processed tensors
- clinical targets

This design is training-ready and reusable across:
- training
- validation
- inference

In [ ]:

class EchoDataset(Dataset):

    def __init__(
        self,
        filelist,
        tracings,
        videos_dir,
        transform=None
    ):

        # Store dataset metadata
        self.filelist = filelist
        self.tracings = tracings
        self.videos_dir = videos_dir

        # Store preprocessing pipeline
        self.transform = transform

        print(
            "Before Filtering:",
            len(self.filelist)
        )

        available_tracings = set(
            self.tracings["FileName"]
        )

        self.filelist = self.filelist[
            self.filelist["FileName"].apply(
                lambda x:
                x + ".avi"
                in available_tracings
            )
        ].reset_index(drop=True)

        print(
            "After Filtering:",
            len(self.filelist)
        )

    def __len__(self):
        return len(self.filelist)

    def __getitem__(self, idx):

        # Retrieve metadata row

        row = self.filelist.iloc[idx]
        video_name,video_path,status,message = validate_row(row)

        if not status:
            raise RuntimeError(f"{video_name}: {message}")

        # Retrieve annotated frames

        frame_counts = get_annotated_frames(
            self.tracings,
            video_name
        )

        annotated_frames = sorted(
            frame_counts.index.tolist()
        )

        if len(annotated_frames) == 0:

            raise RuntimeError(
                f"{video_name}: no annotations found"
            )

        # Approximate ED / ES selection
        ed_idx = annotated_frames[0]
        es_idx = annotated_frames[-1]

        # Read video frames

        cap = cv2.VideoCapture(str(video_path))
        frames = []

        while True:
            ret, frame = cap.read()
            if not ret:
                break
            frame_rgb = cv2.cvtColor(
                frame,
                cv2.COLOR_BGR2RGB
            )
            frames.append(frame_rgb)

        cap.release()

        # Extract ED / ES frames

        ed_frame = frames[ed_idx]
        es_frame = frames[es_idx]

        # Apply transforms
        if self.transform:
            ed_frame = self.transform(ed_frame)
            es_frame = self.transform(es_frame)

        video_tensor = torch.stack([ed_frame,es_frame])

        # Clinical labels

        ef_value = float(row["EF"])
        severity_id = int(row["SeverityID"])
        split = row["Split"]

        # Return training-ready sample

        return {
                "video": video_tensor,
                "ef": ef_value,
                "edv": float(row["EDV"]),
                "esv": float(row["ESV"]),
                "severity": severity_id,
                "video_name": video_name,
                "split": split
        }

# 12. Dataset Construction

## Concept

Instantiate the preprocessing dataset using the desired transform pipeline.

This enables:

- transform modularity
- easy experimentation
- clean separation of preprocessing logic

In [ ]:
print("Tracings sample:")
print(tracings.head(3))
print("Unique videos in tracings:", tracings["FileName"].nunique())

In [ ]:
# Create Dataset Instance

dataset = EchoDataset(
    filelist=filelist,
    tracings=tracings,
    videos_dir=VIDEOS_DIR,
    transform=train_transform
)

print("Dataset Length:",len(dataset))

# 13. Dataset Validation

## Concept

Before building DataLoaders or training loops,
the dataset output should be inspected carefully.

We validate:

- output structure
- tensor shapes
- labels
- metadata fields

In [ ]:
sample = dataset[0]
print("Returned Keys:")
print(sample.keys())

print("Video Shape:",sample["video"].shape)
print("Tensor Type:",sample["video"].dtype)

In [ ]:
# Label Validation
print("EF Value:",sample["ef"])
print("EDV:",sample["edv"])
print("ESV:",sample["esv"])
print("Severity ID:",sample["severity"])
print("Video Name:", sample["video_name"])
print( "Dataset Split:", sample["split"])

# 14.1 Visual Sample Validation

## Concept

Inspecting tensors visually is an important debugging practice.

This helps verify:

- preprocessing correctness
- frame ordering
- ED / ES extraction quality

In [ ]:
video_tensor = sample["video"]

ed_img = (
    video_tensor[0]
    .permute(1,2,0)
    .cpu()
    .numpy()
)

es_img = (
    video_tensor[1]
    .permute(1,2,0)
    .cpu()
    .numpy()
)

plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.imshow((ed_img - ed_img.min())/(ed_img.max() - ed_img.min()))
plt.title("ED Frame")
plt.axis("off")
plt.subplot(1,2,2)
plt.imshow((es_img - es_img.min())/(es_img.max() - es_img.min()))
plt.title("ES Frame")
plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
idx = 330

row = filelist.iloc[idx]
video_name = row["FileName"]

print("Video:", video_name)
fc = get_annotated_frames(tracings, video_name)

print(fc)

frames = sorted(fc.index.tolist())

ed_idx = frames[0]
es_idx = frames[-1]

print("ED Frame:", ed_idx)
print("ES Frame:", es_idx)
video_path = VIDEOS_DIR / f"{video_name}.avi"

cap = cv2.VideoCapture(str(video_path))

# ED
cap.set(cv2.CAP_PROP_POS_FRAMES, ed_idx)
_, frame = cap.read()
ed_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

# ES
cap.set(cv2.CAP_PROP_POS_FRAMES, es_idx)
_, frame = cap.read()
es_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

cap.release()

plt.figure(figsize=(5,5))

plt.subplot(1,2,1)
plt.imshow(ed_frame)
plt.title(f"ED ({ed_idx})")
plt.axis("off")

plt.subplot(1,2,2)
plt.imshow(es_frame)
plt.title(f"ES ({es_idx})")
plt.axis("off")

plt.show()

# 14.2 Multi-Sample Dataset Verification

## Concept

Large-scale visual inspection is a practical debugging strategy.

Visualizing multiple samples helps validate:

- preprocessing correctness
- ED / ES ordering
- dataset quality
- label consistency

In [ ]:
fig, axes = plt.subplots(5,4,figsize=(14,16))

axes = axes.flatten()
indices = np.random.choice(len(dataset),20,replace=False)

for ax, idx in zip(axes,indices):
    print(idx+1)
    sample = dataset[idx]
    img = (sample["video"][0].permute(1,2,0).numpy())
    severity = sample["severity"]
    ef = sample["ef"]
    ax.imshow((img - img.min())/(img.max() - img.min()))
    ax.set_title(f"EF:{ef:.1f} | S:{severity}")
    ax.axis("off")

plt.tight_layout()
plt.show()

# 15. Final Engineering Summary

## Concept

Before progressing toward encoder development,
the preprocessing pipeline should be summarized and validated.

This final checkpoint confirms:

- preprocessing correctness
- batching functionality
- GPU readiness
- label integration

In [ ]:
print("\n03_preprocessing.ipynb completed successfully.")